In [ ]:
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import mlflow
import torch
import torchvision.transforms.v2 as v2
from countcv.core.data import (
	CountingDataset,
	SyncedTransform,
	get_densitymap_loaders,
)
from countcv.core.fcrn_model import FCRNBase, SAUnet
from PIL import Image, ImageChops
from torch.utils.data import DataLoader


In [ ]:
import json
from pathlib import Path
import mlflow


a_path = Path("../mlruns/168285462779286706")
mlflow.set_tracking_uri("../mlruns")
client = mlflow.tracking.MlflowClient()
# experiment_id = "168285462779286706"
experiment_name = "train_size_local"
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id
results = client.search_runs(experiment_id, order_by=["metrics.val_loss"], max_results=2)
print(results.to_list()[0].to_dictionary())

In [ ]:
idx = range(1, 9)
fig, axs = plt.subplots(2, 4, figsize=(16, 9))
img_names = [Path(f"../data/synthetic_cells/raw/00{i}cell.png") for i in idx]
label_names = [Path(f"../data/synthetic_cells/raw/00{i}dots.png") for i in idx]

for img_name, label_name, ax in zip(img_names, label_names, axs.flat):
	image = Image.open(img_name).convert("RGB")
	label = Image.open(label_name).convert("RGB")
	imgshow = ImageChops.add(image, label)
	ax.imshow(imgshow)
	ax.axis("off")
fig.tight_layout()

# Allgemeines Vorgehen

## Ausgangslage

200 synthetische Bilder mit Punktannotationen. 100 davon für Test reserviert

## Ziel: Zellkerne zählen

**Erste Idee:** 
* Bild in neuronales Netz geben mit Regression head, also (3,h,w) -> (1,1,1)
* Eval metric: mean squared error (MSE)

**Zweite Idee:** 
* Bild in neuronales Netz (segmentation) geben und binäre Maske mit Punktannotationen vorhersagen (3,h,w) -> (1,h,w)  
* mit Sigmoid Activation am Ende für Binärität  
* Eval metric: Mask IoU
* Problem: Die Masken sind mini (1 Pixel), daher ist die Eval metrik schwer zu lernen

**Dritte Idee:** 
* Bild in neuroanles Netz (segmentation body) mit Density map (3,h,w) -> (1,h,w)
* Keine Activation am Ende, unconstrain predictions (eventuell ReLU)  
* Eval metric: pixel-wise L2 Loss


## Datenverarbeitung

1. Train/val/test split der Bilder in 64/36/100
2. Falls trainingsdaten: Data augmentation
4. Gaussfilter auf die Punktannotationen

### Data Augmentation (synchron für Ausgangsbild und Density map)

1. Random flips: Horizontal und vertikal
2. 90°, 180°, 270° rotationen (um Interpolation zu vermeiden)
3. RandomResizedCrop, um leichte Verzerrungen einzufügen (bei gleichbleibender Bildgröße)

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(16, 9))
img_tensor = v2.functional.pil_to_tensor(image)
label_tensor = v2.functional.pil_to_tensor(label)

axs[0].imshow(label_tensor[0])
axs[0].axis("off")
axs[0].set_title(f"Count: {label_tensor.sum() / 255}")

alpha = 100
sigma = 2
transform_kwargs = {"size": 256, "alpha": alpha, "sigma": sigma}
transform = SyncedTransform(**transform_kwargs)

img_tensor, blurred_label_tensor = transform(img_tensor, label_tensor, mode="val")

axs[1].imshow(blurred_label_tensor[0] / torch.max(blurred_label_tensor))
axs[1].axis("off")
axs[1].set_title(f"Count: {blurred_label_tensor.sum() / alpha:.2f}")


### Vergleich zu Gaussian kernels mit torch und scipy

In der Mitte von Bildern sind die beiden Ansätze identisch, am Rand verhält sich scipy etwas anders durch den mode="reflect" Parameter. Durch mode="mirror" sind beide Ansätze identisch

In [ ]:
import pandas as pd
import numpy as np

sigmas = torch.arange(1, 10, step=0.1)

max_vals = []
for sigma in sigmas:
	sigma = sigma.item()
	alpha = 25 * sigma**2
	x = torch.zeros((50, 50))
	x[25, 25] = alpha
	kernel_size = 2 * round(4 * sigma) + 1
	gauss_blur = v2.GaussianBlur(kernel_size, sigma)
	torch_x = gauss_blur(x.unsqueeze(0).unsqueeze(0))
	max_vals.append(torch_x.max().item())


df = pd.DataFrame({"sigmas": sigmas.numpy(), "max": max_vals})
df.plot("sigmas", "max")

In [ ]:
from scipy import ndimage

fig, axs = plt.subplots(ncols=3, figsize=(21, 9))

size = 21
sigma = 4
kernel_size = 2 * round(4 * sigma) + 1
alpha = 25 * sigma**2
print(alpha)
gauss_blur = v2.GaussianBlur(kernel_size=kernel_size, sigma=sigma)

x = torch.zeros((size, size))
x[size // 2, size // 2] = 1
x[size // 2 + 1, size // 2 + 1] = 1
x[2, 2] = 1
x = x * alpha
img = axs[0].matshow(x.numpy())

torch_x = gauss_blur(x.unsqueeze(0).unsqueeze(0))
axs[1].matshow(torch_x.squeeze().numpy())
axs[1].set_title("Small gaussian blur")


sigma_l = sigma
kernel_size = 2 * round(4 * sigma_l) + 1
xl = torch.zeros((5 * size, 5 * size))
xl[2 * size, 2 * size] = 4
# xl[size + 2, size + 2] = 4
# xl[4, 4] = 4
xl = xl * alpha
torch_x_large = ndimage.gaussian_filter(xl, sigma=sigma_l)
axs[2].matshow(torch_x_large)
axs[2].set_title("Large gaussian blur")

print(torch_x_large.max())

# Modelltraining

Als Modell eignet sich der Body von jedem Segmentation Modell mit einem "Density map"-head. Das ist z.B eine letzte Convolution mit 1x1 Kernel und einem Output Feature.

## Fully convolutional regression network (FCRN)
<img src="../assets/images/fcrn_model.png" alt="FCRN Modellarchitektur" width="700px" />


## (SA)Unet

<img src="../assets/images/saunet_model.png" alt="SAUnet Modellarchitektur" width="700px" />

**Optimierungsfunktion: Pixelwise MSE-Loss**

## Ergebnisse erster Tests

In [ ]:
alpha = 100
sigma = 2
device = "cuda"
transform_kwargs = {"flip_prob": 0.5, "size": 256, "alpha": alpha, "sigma": sigma}
transform = SyncedTransform(**transform_kwargs)
test_dataset = SyntheticCellDataset(data_root=Path("../data/synthetic_cells"), mode="test", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
n_test = len(test_dataset)

model_path = Path("../models/fcrn-base_SYNTHETIC_CELLS_COUNTING.pt")
model = FCRNBase(input_channels=3)
model.load_state_dict(torch.load(model_path, weights_only=True))
model.to(device)
count, pred_count = torch.zeros(n_test), torch.zeros(n_test)
deviation = torch.zeros(n_test)
for i, (img, label) in enumerate(test_loader):
	with torch.no_grad():
		model.eval()
		img, label = img.to(device), label.to(device)
		outputs = model(img)
		count[i] = label.sum(axis=(1, 2, 3)) / alpha
		pred_count[i] = outputs.sum(axis=(1, 2, 3)) / alpha
		deviation[i] = torch.abs(outputs.sum(axis=(1, 2, 3)) - label.sum(axis=(1, 2, 3))) / alpha


print(f"Avg. number of cells {torch.mean(count):.2f}±{torch.std(count):.2f}")
print(f"Avg. mean absolute error {torch.mean(deviation):.2f}")


| Method                         | MAE         | Ntrain |
|--------------------------------|-------------|--------|
| ResNet-152 (R), Xue et al. 2016 | 7.5 ± 2.2   | 100    |
| GMN, Lu et al. 2018             | 3.6 ± 0.3   | 32     |
| FCRN-A, Xie et al. 2018         | 2.9 ± 0.2   | 64     |
| CCF, Jiang et al., 2020           | 2.6 ± 0.1   | 50     |
| **Count-Ception, Cohen et al., 2017**  | **2.3 ± 0.4**  | **50**     |
| SAU-Net, Guo et al, 2022            | 2.6 ± 0.4   | 64     |
| FCRN-A with improvements (own)  | 2.58 		| 64 |

## Haben wir Bilder die total falsch vorhergesagt werden?

In [ ]:
mincount, maxcount = torch.min(torch.concat([count, pred_count])), torch.max(torch.concat([count, pred_count]))
fig, axs = plt.subplots(ncols=2, figsize=(16, 5))
axs[0].scatter(count, pred_count)
axs[0].axline((mincount, mincount), (maxcount, maxcount), color="black")
axs[0].set_xlabel("True number of cells")
axs[0].set_ylabel("Predicted number of cells")

axs[1].scatter(count, pred_count / count)
axs[1].axline((200, 1), slope=0, color="black")
axs[1].set_xlabel("True number of cells")
axs[1].set_ylabel("Predicted number of cells / True number of cells")
plt.show()

In [ ]:
# Determine global min and max for shared colormap scale
vmin = min(label[0].min(), outputs[0].min()).item()
vmax = max(label[0].max(), outputs[0].max()).item()

# Create figure with custom grid layout
fig = plt.figure(figsize=(15, 8))
gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 0.05])  # last column for colorbar

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2])
cax = fig.add_subplot(gs[3])  # axis for colorbar


denorm_transform = v2.Compose(
	[
		v2.Normalize(torch.tensor([0]), torch.tensor([1 / 0.0030, 1 / 0.0031, 1 / 0.2132])),
		v2.Normalize(torch.tensor([-0.0017, -0.0018, -0.3329]), torch.tensor([1])),
	]
)
img_show = denorm_transform(img.squeeze())
im0 = ax0.imshow(img_show.permute((1, 2, 0)).cpu().numpy())
ax0.set_title("Image")
ax0.axis("off")

# Plot both images with shared color limits
im1 = ax1.imshow(label[0].squeeze().cpu().numpy(), cmap="viridis", vmin=vmin, vmax=vmax)
ax1.set_title(f"Groundtruth | Count: {label[0].sum() / alpha:.02f}")
ax1.axis("off")

im2 = ax2.imshow(outputs[0].squeeze().detach().cpu().numpy(), cmap="viridis", vmin=vmin, vmax=vmax)
ax2.set_title(f"Model output | Count: {outputs[0].sum() / alpha:.02f}")
ax2.axis("off")

# Add single shared colorbar
fig.colorbar(im2, cax=cax)

plt.tight_layout()
plt.show()

In [ ]:
x = outputs[0]


def summary(x):
	return {
		"min": x.min().item(),
		"max": x.max().item(),
		"mean": x.mean().item(),
		"std": x.std(unbiased=False).item(),  # unbiased=False for population std
		"q25": torch.quantile(x, 0.25).item(),
		"median": torch.median(x).item(),
		"q75": torch.quantile(x, 0.75).item(),
	}


print(f"Pred        {summary(x)}")
print(f"Groundtruth {summary(label[0])}")

Wo liegt das Modell eigentlich in einem Bild falsch? Im nächsten Bild sieht man, dass häufig die genaue Lokalisierung der Annotation schwierig ist, aber die Zelle durchaus gefunden wird

In [ ]:
diff = outputs[0] - label[0]

plt.imshow(diff.squeeze().detach().cpu().numpy(), cmap="viridis", vmin=-1, vmax=1)
plt.colorbar()

# CarPK Datensatz

Als weitere Untersuchung schauen wir uns die CarPK Daten aus https://lafi.github.io/LPN/ 

Warum haben wir diesen Datensatz ausgewählt?  

1. Normale Fotos (keine Mikroskopie, keine Satellitenbilder) --> Vermutlich gut für YOLO, GroundingDINO
2. Sehr hohe Datenqualität. 1700 Bilder händisch gelabelt mit ca. 90k Autos
3. Bounding boxes sind bereitgestellt und lassen sich gut in Densitymaps übersetzen

In [ ]:
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.v2 as v2
from PIL import Image, ImageChops
from torch.utils.data import DataLoader
from torchvision.transforms import InterpolationMode

from effcv.dataloaders import CountingDataset, SyncedTransform, get_densitymap_loaders
from effcv.fcrn_model import SAUnet
from effcv.experiment_utils import calc_metrics


In [ ]:
data_dir = Path("../data/carpk")
dataset_dmaps = CountingDataset(
	data_dir,
	set_type="val",
	fetch_dot_labels=True,
)

img, densitymap = dataset_dmaps[20]

In [ ]:
dataset_bbox = CountingDataset(data_dir, set_type="val", fetch_dot_labels=False)
dataset_dmaps = CountingDataset(
	data_dir, set_type="val", fetch_dot_labels=True, transform=SyncedTransform(size=[512, 256], sigma=2)
)
img, label = dataset_bbox[20]
img2, densitymap = dataset_dmaps[20]

In [ ]:
from PIL import ImageDraw

pil_img = v2.functional.to_pil_image(img)
draw = ImageDraw.Draw(pil_img)
for box in label["bboxes"]:
	x1, y1, x2, y2 = box
	draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
pil_img

In [ ]:
pil_dmap = v2.functional.to_pil_image(densitymap)
draw = ImageDraw.Draw(pil_dmap)
for box in label["bboxes"]:
	x1, y1, x2, y2 = box
	draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
pil_dmap

In [ ]:
device = "cuda"
batch_size = 8
transform_kwargs = {
	"size": (512, 256),
	"alpha": 100,
	"sigma": 2,
}

train_loader, val_loader = get_densitymap_loaders(
	batch_size=batch_size, data_dir=data_dir, transform_kwargs=transform_kwargs
)

In [ ]:
alpha = 100
model_path = Path("../checkpoints/SAUnet_densitymaps_tests.pt")
model = SAUnet(input_channels=3)
model.load_state_dict(torch.load(model_path))
model.to(device)
model.eval()
counts = torch.tensor([], dtype=torch.float, device=device)
pred_counts = torch.tensor([], dtype=torch.float, device=device)

for i, (img, label) in enumerate(val_loader):
	with torch.no_grad():
		img, label = img.to(device), label.to(device)
		outputs = model(img)
		counts = torch.concat((counts, label.sum(axis=(1, 2, 3)) / alpha))
		pred_counts = torch.concat((pred_counts, outputs.sum(axis=(1, 2, 3)) / alpha))

calc_metrics(pred_counts.cpu(), counts.cpu())

In [ ]:
mincount, maxcount = torch.min(torch.concat([count, pred_count])), torch.max(torch.concat([count, pred_count]))
fig, axs = plt.subplots(ncols=2, figsize=(16, 5))
axs[0].scatter(count, pred_count)
axs[0].axline((mincount, mincount), (maxcount, maxcount), color="black")
axs[0].set_xlabel("True number of cars")
axs[0].set_ylabel("Predicted number of cars")

axs[1].scatter(count, pred_count / count)
axs[1].axline((100, 1), slope=0, color="black")
axs[1].set_xlabel("True number of cars")
axs[1].set_ylabel("Predicted number of cars / True number of cars")
plt.show()


In [ ]:
v2.functional.to_pil_image(v2.functional.resize(img[1], size=(720, 1280)))

In [ ]:
diff = outputs[1] - label[1]

plt.imshow(diff.squeeze().detach().cpu().numpy(), cmap="viridis", vmin=-1, vmax=1)
plt.colorbar()


In [ ]:
import sys

sys.path.append("..")

from countcv.core.experiment_utils import load_mlflow_model
from countcv.core.data import CountingDataset
from pathlib import Path
from torchvision.transforms import v2

data_dir = Path("../data/uc_cells")
tracking_uri = "../mlruns"
run_id = "e3b28d01aee742e0a3c3b4df2ebfab9f"
model, transform = load_mlflow_model(tracking_uri, run_id)
dataset_dmaps = CountingDataset(
	data_dir,
	set_type="val",
	fetch_dot_labels=True,
	transform=transform,
)

In [ ]:
img, label = dataset_dmaps[1]
img = img.to("cuda")
pred = model.predict(img.unsqueeze(0), device="cuda")
v2.functional.to_pil_image((pred.squeeze() - pred.min()) / pred.max())


In [ ]:
from pathlib import Path

import pandas as pd


def calc_statistics(df: pd.DataFrame):
	stats = df.copy()
	predtypes = ["yolo", "density"]
	for pred in predtypes:
		stats[f"{pred}_mae"] = (stats["gt"] - stats[pred]).abs()
		stats[f"{pred}_intervalsize"] = stats[f"{pred}_interval_max"].round() - stats[f"{pred}_interval_min"].round()
		stats[f"{pred}_coverage"] = (stats["gt"] <= stats[f"{pred}_interval_max"].round()) & (
			stats["gt"] >= stats[f"{pred}_interval_min"].round()
		)
	return stats


In [ ]:
df = pd.read_csv(Path("../testout/2026-01-27_11-21-27/results.csv"), delimiter=";", decimal=",")
df = calc_statistics(df)
colnames = ["yolo_mae", "density_mae", "yolo_intervalsize", "density_intervalsize", "yolo_coverage", "density_coverage"]

for bins in [1, 2, 6]:
	grouped = df.groupby(pd.qcut(df["gt"], bins), observed=True)

	res = grouped[colnames].mean()
	res["count"] = grouped.size()
	print(res.to_string())
	print(2 * "\n")